# Probability Bowl Model v2 - World Cup 2026
Fill in HOME, AWAY, referee yellows. Everything else is automatic.

In [ ]:
# CELL 1: Setup - run once per session
!pip install scipy numpy pandas requests --quiet
import numpy as np, pandas as pd, json, urllib.request
from scipy.stats import poisson
from scipy.optimize import minimize
import warnings; warnings.filterwarnings('ignore')
np.random.seed(42)
N_SIMS = 300_000
print('Ready.')

In [ ]:
# ============================================================
# ONE-CELL MATCH RUNNER - fill in 3 things only
# ============================================================

HOME = "Netherlands"
AWAY = "Morocco"
REFEREE_YELLOWS_PER_GAME = 2.5  # Sampaio - HIGHLY VOLATILE: 3 reds in match 1, 0 cards in match 2
SUSPENSION_NOTE = "None confirmed - Morocco rotating back to strongest XI (Hakimi, Bouaddi, Diaz, Mazraoui all returning)"

# ---- EVERYTHING BELOW IS AUTOMATIC -------------------------

import numpy as np, pandas as pd, json, urllib.request
from scipy.stats import poisson
from scipy.optimize import minimize

print(f"\n{'='*55}")
print(f"  {HOME} vs {AWAY}")
print(f"{'='*55}")

# STEP 1: Auto-fetch live odds (optional - paste key if you have one)
ODDS_API_KEY = ""

def fetch_odds(home, away, api_key):
    if not api_key:
        return None
    try:
        url = (f"https://api.the-odds-api.com/v4/sports/soccer_fifa_world_cup"
               f"/odds/?apiKey={api_key}&regions=us&markets=h2h,totals&oddsFormat=american")
        with urllib.request.urlopen(url, timeout=8) as r:
            games = json.loads(r.read())
        for g in games:
            teams = [g["home_team"], g["away_team"]]
            if any(home.lower() in t.lower() for t in teams) and any(away.lower() in t.lower() for t in teams):
                return g
        return None
    except Exception as e:
        print(f"  Odds fetch failed: {e}")
        return None

odds_data = fetch_odds(HOME, AWAY, ODDS_API_KEY)

# STEP 2: Auto-fetch WC 2026 team rates from martj42
print("\n  Loading WC 2026 team rates...")
results = pd.read_csv("https://raw.githubusercontent.com/martj42/international_results/master/results.csv")
scorers = pd.read_csv("https://raw.githubusercontent.com/martj42/international_results/master/goalscorers.csv")

wc26 = results[
    results["tournament"].str.contains("FIFA World Cup", na=False) &
    (results["date"] >= "2026-06-01") &
    results["home_score"].notna() &
    (results["home_score"] != "NA")
].copy()
wc26["home_score"] = pd.to_numeric(wc26["home_score"], errors="coerce")
wc26["away_score"] = pd.to_numeric(wc26["away_score"], errors="coerce")
wc26 = wc26.dropna(subset=["home_score","away_score"])

h_rec = wc26[["home_team","home_score","away_score"]].rename(columns={"home_team":"team","home_score":"scored","away_score":"conceded"})
a_rec = wc26[["away_team","away_score","home_score"]].rename(columns={"away_team":"team","away_score":"scored","home_score":"conceded"})
all_rec = pd.concat([h_rec, a_rec])
wc26_team = all_rec.groupby("team").agg(gf=("scored","mean"), ga=("conceded","mean"), mp=("scored","count")).reset_index()
WC26_AVG = wc26_team["gf"].mean()

def team_rate(team):
    row = wc26_team[wc26_team["team"] == team]
    if len(row) == 0: return {"gf": WC26_AVG, "ga": WC26_AVG, "mp": 0}
    r = row.iloc[0]
    return {"gf": round(r["gf"],3), "ga": round(r["ga"],3), "mp": int(r["mp"])}

hp = team_rate(HOME)
ap = team_rate(AWAY)
print(f"  {HOME}: {hp['gf']} GF/g, {hp['ga']} GA/g ({hp['mp']} games)")
print(f"  {AWAY}: {ap['gf']} GF/g, {ap['ga']} GA/g ({ap['mp']} games)")

# STEP 3: Compute lambdas
lam_hist = (hp["gf"] / WC26_AVG) * (ap["ga"] / WC26_AVG) * WC26_AVG
mu_hist  = (ap["gf"] / WC26_AVG) * (hp["ga"] / WC26_AVG) * WC26_AVG

if odds_data:
    h_ml, a_ml, d_ml = -110, 110, 220
    for book in odds_data.get("bookmakers", [])[:1]:
        for mkt in book.get("markets", []):
            if mkt["key"] == "h2h":
                outcomes = {o["name"]: o["price"] for o in mkt["outcomes"]}
                h_ml = outcomes.get(odds_data["home_team"], h_ml)
                a_ml = outcomes.get(odds_data["away_team"], a_ml)
                d_ml = outcomes.get("Draw", d_ml)
    def ai(o): return 100/(o+100) if o>0 else abs(o)/(abs(o)+100)
    raw = {"h": ai(h_ml), "d": ai(d_ml), "a": ai(a_ml)}
    tot_v = sum(raw.values())
    fair = {k: v/tot_v for k,v in raw.items()}
    print(f"\n  Market: Home {fair['h']:.1%} Draw {fair['d']:.1%} Away {fair['a']:.1%}")

    def dc_obj(params):
        lamv, muv = params
        if lamv<=0 or muv<=0: return 1e6
        rho=-0.13
        def tau(x,y):
            if x==0 and y==0: return 1-lamv*muv*rho
            if x==0 and y==1: return 1+lamv*rho
            if x==1 and y==0: return 1+muv*rho
            if x==1 and y==1: return 1-rho
            return 1.0
        M = np.array([[tau(i,j)*poisson.pmf(i,lamv)*poisson.pmf(j,muv) for j in range(8)] for i in range(8)])
        M /= M.sum()
        return (np.tril(M,-1).sum()-fair["h"])**2 + (np.diag(M).sum()-fair["d"])**2 + (np.triu(M,1).sum()-fair["a"])**2

    avg_g = (lam_hist + mu_hist)
    ratio = (fair["h"]+0.5*fair["d"])/(fair["a"]+0.5*fair["d"]+1e-6)
    lam0 = avg_g*ratio/(1+ratio); mu0 = avg_g-lam0
    res = minimize(dc_obj, [lam0,mu0], bounds=[(0.1,5),(0.1,5)], method="L-BFGS-B")
    lam_mkt, mu_mkt = res.x
    lam = 0.65*lam_mkt + 0.35*lam_hist
    mu  = 0.65*mu_mkt  + 0.35*mu_hist
else:
    lam, mu = lam_hist, mu_hist
    print(f"\n  Using WC26 historical rates only (no odds API key set)")

print(f"  Lambda: home={lam:.3f} away={mu:.3f}")

# STEP 4: Auto-set card/corner/offside/SOT rates from team history
cards_per_team = REFEREE_YELLOWS_PER_GAME / 2
cards_home = round(cards_per_team * 1.0, 2)
cards_away = round(cards_per_team * 1.0, 2)

corners_home = round(3 + hp["gf"] * 1.2, 1)
corners_away = round(3 + ap["gf"] * 1.2, 1)

offsides_home = round(0.8 + hp["gf"] * 0.5, 1)
offsides_away = round(0.8 + ap["gf"] * 0.5, 1)

# SOT rate proxy: goal rate scaled up (roughly 3.2x goals = SOT, conversion ~30%)
shots_home = round(max(lam, hp["gf"]) * 3.2, 1)
shots_away = round(max(mu, ap["gf"]) * 3.2, 1)

print(f"\n  Auto-computed rates:")
print(f"  Cards:    home={cards_home} away={cards_away} (ref avg={REFEREE_YELLOWS_PER_GAME})")
print(f"  Corners:  home={corners_home} away={corners_away}")
print(f"  Offsides: home={offsides_home} away={offsides_away}")
print(f"  SOT:      home={shots_home} away={shots_away}")

# STEP 5: Simulate
print("\n  Running 300K simulations...")
n = 300_000; rho = -0.13
h1=np.random.poisson(lam*0.45,n); a1=np.random.poisson(mu*0.45,n)
h2=np.random.poisson(lam*0.55,n); a2=np.random.poisson(mu*0.55,n)
hg=h1+h2; ag=a1+a2
mask=(hg==0)&(ag==0); rej=np.random.random(n)>(1-lam*mu*rho); rs=mask&rej
hg[rs]=np.random.poisson(lam,rs.sum()); ag[rs]=np.random.poisson(mu,rs.sum())
ch=np.random.poisson(cards_home,n); ca=np.random.poisson(cards_away,n)
kh=np.random.poisson(corners_home,n); ka=np.random.poisson(corners_away,n)
oh=np.random.poisson(offsides_home,n); oa=np.random.poisson(offsides_away,n)
sh=np.random.poisson(shots_home,n); sa=np.random.poisson(shots_away,n)
tot=hg+ag; g1h=h1+a1; g2h=h2+a2; tc=ch+ca
btts=(hg>=1)&(ag>=1)

# Hydration break + sub goal props
frac_hb1 = 30/90
frac_hb2 = (90-75)/90
oh_hb1=np.random.poisson(offsides_home*frac_hb1,n)
oa_hb1=np.random.poisson(offsides_away*frac_hb1,n)
gh_hb1=np.random.poisson(lam*frac_hb1,n); ga_hb1=np.random.poisson(mu*frac_hb1,n)
ch_hb2=np.random.poisson(cards_home*frac_hb2,n); ca_hb2=np.random.poisson(cards_away*frac_hb2,n)
sub_g=np.random.poisson((lam+mu)*0.25*0.20,n)

results_dict = {
    f"{HOME} win":           (hg>ag).mean(),
    "Draw":                  (hg==ag).mean(),
    f"{AWAY} win":           (hg<ag).mean(),
    f"{HOME} score 1+":      (hg>=1).mean(),
    f"{AWAY} score 1+":      (ag>=1).mean(),
    "BTTS":                  btts.mean(),
    "Over 1.5 goals":        (tot>=2).mean(),
    "Over 2.5 goals":        (tot>=3).mean(),
    "Under 2 goals":         (tot<=2).mean(),
    "3+ total goals":        (tot>=3).mean(),
    "BTTS + 3+ goals":       (btts&(tot>=3)).mean(),
    "HT draw":               (h1==a1).mean(),
    f"{HOME} leading HT":    (h1>a1).mean(),
    "2H more goals than 1H": (g2h>g1h).mean(),
    "2H has 2+ goals":       (g2h>=2).mean(),
    f"{HOME} score in 2H":   (h2>=1).mean(),
    f"{AWAY} score in 2H":   (a2>=1).mean(),
    f"{HOME} score in both halves": ((h1>=1)&(h2>=1)).mean(),
    "3+ total cards":        (tc>=3).mean(),
    "4+ total cards":        (tc>=4).mean(),
    f"{HOME} more cards":    (ch>ca).mean(),
    f"{AWAY} more cards":    (ca>ch).mean(),
    f"{HOME} 5+ corners":    (kh>=5).mean(),
    f"{AWAY} 5+ corners":    (ka>=5).mean(),
    f"{HOME} 7+ corners":    (kh>=7).mean(),
    f"{HOME} more corners":  (kh>ka).mean(),
    f"{AWAY} more corners":  (ka>kh).mean(),
    "9+ total corners":      ((kh+ka)>=9).mean(),
    f"{HOME} offside 2+":    (oh>=2).mean(),
    f"{AWAY} offside 2+":    (oa>=2).mean(),
    "3+ total offsides":     ((oh+oa)>=3).mean(),
    "Offside before 1st break (30min)": ((oh_hb1>=1)|(oa_hb1>=1)).mean(),
    "Goal before 1st break (30min)":    ((gh_hb1+ga_hb1)>=1).mean(),
    "Card after 2nd break (75min)":     ((ch_hb2+ca_hb2)>=1).mean(),
    "Substitute scores":     (sub_g>=1).mean(),
    f"{HOME} 7+ SOT":        (sh>=7).mean(),
    "8+ total SOT":          ((sh+sa)>=8).mean(),
}

CAL = {"4+ total cards": 0.72, "3+ total cards": 0.80, "2H more goals than 1H": 0.87}
for k in list(results_dict.keys()):
    if k in CAL:
        results_dict[k+" [calibrated]"] = float(np.clip(results_dict[k]*CAL[k],0.05,0.95))

print("\n  ALL PROPS:")
out = pd.DataFrame([{"Prop": k, "Prob": f"{v:.1%}"} for k,v in results_dict.items()])
print(out.to_string(index=False))


In [ ]:
# CELL 3: Player props - fill in per match
def player_goal_prob(name, goals_per_game=None, mins=90, role=1.0):
    sc = pd.read_csv('https://raw.githubusercontent.com/martj42/international_results/master/goalscorers.csv')
    sc26 = sc[(sc['date']>='2026-06-01') & (~sc['own_goal'].isin([True,'True']))]
    ps = sc26.groupby('scorer').agg(g=('scorer','count'),m=('date','nunique')).reset_index()
    ps['gpg'] = ps['g']/ps['m']
    row = ps[ps['scorer'].str.contains(name, case=False, na=False)]
    if len(row) > 0 and goals_per_game is None:
        r = row.iloc[0]
        goals_per_game = r['gpg']
        print(f'  {name}: WC26 rate = {int(r["g"])} goals in {int(r["m"])} games ({goals_per_game:.3f}/g)')
    elif goals_per_game is None:
        print(f'  {name}: no WC26 data, using baseline 0.3/g')
        goals_per_game = 0.3
    lam = goals_per_game * (mins/90) * role
    p1 = 1 - np.exp(-lam)
    p2 = 1 - np.exp(-lam) - lam*np.exp(-lam)
    print(f'  {name}: P(1+ goal)={p1:.1%}  P(2+ goals)={p2:.1%}')
    return p1, p2

def player_sot_prob(name, shots_per_90, sot_acc=0.36, mins=90, role=1.0, context=1.0):
    lam = shots_per_90 * sot_acc * (mins/90) * role * context
    p = 1 - np.exp(-lam)
    print(f'  {name}: P(>=1 SOT) = {p:.1%}')
    return p

print('Player props for Netherlands vs Morocco:\n')

# Brian Brobbey - 3 goals already this tournament, focal point of Dutch attack
player_goal_prob('Brobbey', role=1.0)

# Cody Gakpo
player_goal_prob('Gakpo', role=0.95)

# Achraf Hakimi - sensational form, attacking fullback (lower role weight, more crosses than shots)
player_sot_prob('Achraf Hakimi', shots_per_90=2.0, role=0.65, context=1.0)

# Brahim Diaz - creative playmaker, not primary shooter
player_sot_prob('Brahim Diaz', shots_per_90=2.2, role=0.75, context=1.0)

# Ismael Saibari - 3 goals in 3 games, leading the line for Morocco
player_goal_prob('Saibari', role=1.0)

# Ayyoub Bouaddi - young midfield sensation, ran the show vs Brazil
player_sot_prob('Ayyoub Bouaddi', shots_per_90=1.8, role=0.65, context=1.05)
